In [ ]:
import numpy as np
import pandas as pd

In [4]:
df=pd.read_csv(r'/content/diabetes.csv')
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [5]:
df.shape

(768, 9)

In [10]:
df.corr()['Outcome']

,Outcome
Pregnancies,0.221898
Glucose,0.466581
BloodPressure,0.065068
SkinThickness,0.074752
Insulin,0.130548
BMI,0.292695
DiabetesPedigreeFunction,0.173844
Age,0.238356
Outcome,1.000000


In [16]:
X=df.iloc[:,0:-1]
y=df.iloc[:,-1]

In [18]:
X

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age
0,6,148,72,35,0,33.6,0.627,50
1,1,85,66,29,0,26.6,0.351,31
2,8,183,64,0,0,23.3,0.672,32
3,1,89,66,23,94,28.1,0.167,21
4,0,137,40,35,168,43.1,2.288,33
...,...,...,...,...,...,...,...,...
763,10,101,76,48,180,32.9,0.171,63
764,2,122,70,27,0,36.8,0.340,27
765,5,121,72,23,112,26.2,0.245,30
766,1,126,60,0,0,30.1,0.349,47


### Scaling using StandardScaler

In [19]:
from sklearn.preprocessing import StandardScaler
scaler=StandardScaler()
X=scaler.fit_transform(X)

In [22]:
X

array([[ 0.63994726,  0.84832379,  0.14964075, ...,  0.20401277,
         0.46849198,  1.4259954 ],
       [-0.84488505, -1.12339636, -0.16054575, ..., -0.68442195,
        -0.36506078, -0.19067191],
       [ 1.23388019,  1.94372388, -0.26394125, ..., -1.10325546,
         0.60439732, -0.10558415],
       ...,
       [ 0.3429808 ,  0.00330087,  0.14964075, ..., -0.73518964,
        -0.68519336, -0.27575966],
       [-0.84488505,  0.1597866 , -0.47073225, ..., -0.24020459,
        -0.37110101,  1.17073215],
       [-0.84488505, -0.8730192 ,  0.04624525, ..., -0.20212881,
        -0.47378505, -0.87137393]])

# 1. Now apply Deep Learning Model

In [23]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

In [150]:
import tensorflow
from tensorflow import keras
from keras import Sequential
from keras.layers import Dense ,Dropout

In [39]:

model=keras.Sequential()

model.add(Dense(8,activation='relu',input_dim=8))
model.add(Dense(1,activation='sigmoid'))

model.compile(optimizer='Adam',loss='binary_crossentropy',metrics=['accuracy'])
model.fit(X_train,y_train,batch_size=32,epochs=15,validation_data=(X_test,y_test))

Epoch 1/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.6384 - loss: 0.6961 - val_accuracy: 0.6169 - val_loss: 0.7453
Epoch 2/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6645 - loss: 0.6614 - val_accuracy: 0.6169 - val_loss: 0.7117
Epoch 3/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6743 - loss: 0.6340 - val_accuracy: 0.6234 - val_loss: 0.6835
Epoch 4/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6906 - loss: 0.6130 - val_accuracy: 0.6169 - val_loss: 0.6617
Epoch 5/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7052 - loss: 0.5953 - val_accuracy: 0.6299 - val_loss: 0.6448
Epoch 6/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7068 - loss: 0.5811 - val_accuracy: 0.6429 - val_loss: 0.6314
Epoch 7/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7101 - loss: 0.5697 - val_accuracy: 0.6558 - val_loss: 0.6180
Epoch 8/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7134 - loss: 0.5600 - val_accuracy: 0.6558 - val_loss

# 2. How to select appropriate best optimizer

In [57]:
import kerastuner as kt

In [60]:

def build_model(hp):
  model=Sequential()

  model.add(Dense(32,activation='relu',input_dim=8))
  model.add(Dense(1,activation='sigmoid'))

  optimizer=hp.Choice('optimizer',values=['sgd','adam','rmsprop','adadelta'])

  model.compile(optimizer=optimizer,loss='binary_crossentropy',metrics=['accuracy'])
  return model

In [61]:
tuner=kt.RandomSearch(build_model,objective='val_accuracy',max_trials=5)
tuner.search(X_train,y_train,epochs=10,validation_data=(X_test,y_test))

Trial 4 Complete [00h 00m 04s]
val_accuracy: 0.7402597665786743

Best val_accuracy So Far: 0.7792207598686218
Total elapsed time: 00h 00m 16s


In [84]:
tuner.get_best_hyperparameters()[0].values

{'optimizer': 'adam'}

In [86]:
best_model=tuner.get_best_models(num_models=1)[0]

In [87]:
best_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 32)             │           288 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 321 (1.25 KB)

 Trainable params: 321 (1.25 KB)

 Non-trainable params: 0 (0.00 B)

In [88]:
model.fit(X_train,y_train,epochs=10,batch_size=32,validation_data=(X_test,y_test))

Epoch 1/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.7427 - loss: 0.5148 - val_accuracy: 0.6818 - val_loss: 0.5605
Epoch 2/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - accuracy: 0.7427 - loss: 0.5106 - val_accuracy: 0.6818 - val_loss: 0.5578
Epoch 3/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.7443 - loss: 0.5072 - val_accuracy: 0.6818 - val_loss: 0.5542
Epoch 4/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7459 - loss: 0.5039 - val_accuracy: 0.6883 - val_loss: 0.5522
Epoch 5/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7427 - loss: 0.5006 - val_accuracy: 0.6948 - val_loss: 0.5501
Epoch 6/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7443 - loss: 0.4977 - val_accuracy: 0.6948 - val_loss: 0.5485
Epoch 7/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7508 - loss: 0.4947 - val_accuracy: 0.6948 - val_loss: 0.5467
Epoch 8/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7541 - loss: 0.4925 - val_accuracy: 0.7013 - val_lo

# 3. How to choose best no.of neurons with Fixed Steps

In [90]:
import kerastuner as kt

In [117]:

def build_model(hp):
  model=Sequential()

  units=hp.Int('units',min_value=8,max_value=256,step=8) # fixed steps
  model.add(Dense(units=units,activation='relu',input_dim=8))
  model.add(Dense(1,activation='sigmoid'))

  model.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])
  return model

In [118]:
tuner=kt.RandomSearch(build_model,
                      objective='val_accuracy',
                      max_trials=5,
                      directory='mydir',
                      project_name='NO.of neurons')
tuner.search(X_train,y_train,epochs=5,validation_data=(X_test,y_test))

Reloading Tuner from mydir/NO.of neurons/tuner0.json


In [119]:
tuner.get_best_hyperparameters()[0].values

{'units': 128}

In [120]:
model=tuner.get_best_models(num_models=1)[0]

In [121]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │         1,152 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,281 (5.00 KB)

 Trainable params: 1,281 (5.00 KB)

 Non-trainable params: 0 (0.00 B)

# How to choose best no.of neurons with Random Steps

In [122]:

def build_model(hp):
  model=Sequential()

  units=hp.Int('units',min_value=8,max_value=256) # Random steps
  model.add(Dense(units=units,activation='relu',input_dim=8))
  model.add(Dense(1,activation='sigmoid'))

  model.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])
  return model

In [123]:
tuner=kt.RandomSearch(build_model,
                      objective='val_accuracy',
                      max_trials=5,
                      directory='mydir',
                      project_name='NO.of neurons')
tuner.search(X_train,y_train,epochs=5,validation_data=(X_test,y_test))

Reloading Tuner from mydir/NO.of neurons/tuner0.json


In [124]:
tuner.get_best_hyperparameters()[0].values

{'units': 128}

In [125]:
model=tuner.get_best_models(num_models=1)[0]

In [126]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │         1,152 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,281 (5.00 KB)

 Trainable params: 1,281 (5.00 KB)

 Non-trainable params: 0 (0.00 B)

In [127]:
model.fit(X_train,y_train,epochs=10,batch_size=32,validation_data=(X_test,y_test))

Epoch 1/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.7687 - loss: 0.4793 - val_accuracy: 0.7662 - val_loss: 0.5063
Epoch 2/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7769 - loss: 0.4649 - val_accuracy: 0.7727 - val_loss: 0.5047
Epoch 3/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7769 - loss: 0.4568 - val_accuracy: 0.7792 - val_loss: 0.5067
Epoch 4/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7736 - loss: 0.4525 - val_accuracy: 0.7792 - val_loss: 0.5076
Epoch 5/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7834 - loss: 0.4467 - val_accuracy: 0.7597 - val_loss: 0.5117
Epoch 6/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7850 - loss: 0.4429 - val_accuracy: 0.7597 - val_loss: 0.5120
Epoch 7/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7818 - loss: 0.4401 - val_accuracy: 0.7597 - val_loss: 0.5132
Epoch 8/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7850 - loss: 0.4378 - val_accuracy: 0.7597 - val_loss

### Now we are getting better accuracy and val_accuracy and reduce loss

# 4. How to select NO.of Layers in model

In [128]:
def build_model(hp):
  model=Sequential()

  model.add(Dense(128,activation='relu',input_dim=8))
 # run for i=1 , 1 time and  i=2, 2 times and .... i=10 , 10-times
  for i in range(hp.Int('num_layers',min_value=1,max_value=10)):
    model.add(Dense(128,activation='relu'))
  model.add(Dense(1,activation='sigmoid'))

  model.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])
  return model

In [129]:
tuner=kt.RandomSearch(build_model,
                      objective='val_accuracy',
                      max_trials=5,
                      directory='mydir',
                      project_name='NO.of layers')
tuner.search(X_train,y_train,epochs=5,validation_data=(X_test,y_test))

Trial 5 Complete [00h 00m 04s]
val_accuracy: 0.7857142686843872

Best val_accuracy So Far: 0.7922077775001526
Total elapsed time: 00h 00m 26s


In [130]:
tuner.get_best_hyperparameters()[0].values

{'num_layers': 8}

In [133]:
model=tuner.get_best_models(num_models=1)[0]

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.13/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 42 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [134]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │         1,152 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 133,377 (521.00 KB)

 Trainable params: 133,377 (521.00 KB)

 Non-trainable params: 0 (0.00 B)

In [136]:
model.fit(X_train,y_train,epochs=10,batch_size=32,validation_data=(X_test,y_test))

Epoch 1/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.8322 - loss: 0.3740 - val_accuracy: 0.7662 - val_loss: 0.6389
Epoch 2/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.8322 - loss: 0.3661 - val_accuracy: 0.7143 - val_loss: 0.5898
Epoch 3/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - accuracy: 0.8436 - loss: 0.3431 - val_accuracy: 0.7273 - val_loss: 0.6083
Epoch 4/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.8583 - loss: 0.3200 - val_accuracy: 0.6558 - val_loss: 0.7051
Epoch 5/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7997 - loss: 0.3864 - val_accuracy: 0.7208 - val_loss: 0.6478
Epoch 6/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.8388 - loss: 0.3310 - val_accuracy: 0.7013 - val_loss: 0.6104
Epoch 7/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.8664 - loss: 0.2928 - val_accuracy: 0.7078 - val_loss: 0.8103
Epoch 8/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.8534 - loss: 0.2931 - val_accuracy: 0.6818 - val_lo

### Here we get best accuracy but val_accuracy is decrease which is not good

# 5. Now we apply the Final DL Model with all tasks together and without dropout for better results

In [161]:

def build_model(hp):
  model=Sequential()

  counter=0
  # run for i=1 , 1 time and  i=2, 2 times and .... i=10 , 10-times
  for i in range(hp.Int('num_layers',min_value=1,max_value=10)):

    if counter==0:
        model.add(Dense(hp.Int('units'+str(i),min_value=8,max_value=256,step=8),
                      activation=hp.Choice(f'activation_{i}', values=['relu', 'tanh', 'sigmoid']),
                      input_dim=8)
                )
    else:
        model.add(Dense(hp.Int('units'+str(i),min_value=8,max_value=256,step=8),
                        activation=hp.Choice(f'activation_{i}', values=['relu', 'tanh', 'sigmoid']))
                    )

    counter+=1
  model.add(Dense(1,activation='sigmoid'))
  optimizer=hp.Choice('optimizer',values=['sgd','adam','rmsprop','adadelta'])

  model.compile(optimizer=optimizer,
                loss='binary_crossentropy',metrics=['accuracy'])
  return model

In [162]:
tuner=kt.RandomSearch(build_model,
                      objective='val_accuracy',
                      max_trials=5,
                      directory='mydir',
                      project_name='Final without dropout')
tuner.search(X_train,y_train,epochs=5,validation_data=(X_test,y_test))

Trial 5 Complete [00h 00m 03s]
val_accuracy: 0.7272727489471436

Best val_accuracy So Far: 0.7272727489471436
Total elapsed time: 00h 04m 44s


In [163]:
tuner.get_best_hyperparameters()[0].values

{'num_layers': 1,
 'units0': 112,
 'activation': 'sigmoid',
 'optimizer': 'rmsprop',
 'units1': 200,
 'activation_0': 'tanh',
 'activation_1': 'relu',
 'units2': 96,
 'activation_2': 'tanh',
 'units3': 40,
 'activation_3': 'tanh',
 'units4': 40,
 'activation_4': 'sigmoid',
 'units5': 248,
 'activation_5': 'sigmoid',
 'units6': 184,
 'activation_6': 'relu',
 'units7': 40,
 'activation_7': 'relu',
 'units8': 240,
 'activation_8': 'sigmoid'}

In [164]:
model1=tuner.get_best_models(num_models=1)[0]

/usr/local/lib/python3.13/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 2 variables whereas the saved optimizer has 6 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [165]:
model1.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 112)            │         1,008 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           113 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,121 (4.38 KB)

 Trainable params: 1,121 (4.38 KB)

 Non-trainable params: 0 (0.00 B)

In [179]:
model1.fit(X_train,y_train,epochs=10,batch_size=32,validation_data=(X_test,y_test))

Epoch 1/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.7508 - loss: 0.5053 - val_accuracy: 0.7208 - val_loss: 0.5099
Epoch 2/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7671 - loss: 0.4797 - val_accuracy: 0.7468 - val_loss: 0.4977
Epoch 3/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7606 - loss: 0.4717 - val_accuracy: 0.7403 - val_loss: 0.4956
Epoch 4/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7704 - loss: 0.4692 - val_accuracy: 0.7468 - val_loss: 0.5000
Epoch 5/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7818 - loss: 0.4658 - val_accuracy: 0.7338 - val_loss: 0.5045
Epoch 6/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7704 - loss: 0.4656 - val_accuracy: 0.7662 - val_loss: 0.5000
Epoch 7/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7769 - loss: 0.4653 - val_accuracy: 0.7662 - val_loss: 0.5039
Epoch 8/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7785 - loss: 0.4652 - val_accuracy: 0.7403 - val_loss

# Now we apply the Final DL Model with all tasks together and with dropout for better results

In [169]:

def build_model(hp):
  model=Sequential()

  counter=0
  # run for i=1 , 1 time and  i=2, 2 times and .... i=10 , 10-times
  for i in range(hp.Int('num_layers',min_value=1,max_value=10)):

    if counter==0:
        model.add(Dense(hp.Int('units'+str(i),min_value=8,max_value=256,step=8),
                      activation=hp.Choice(f'activation_{i}', values=['relu', 'tanh', 'sigmoid']),
                      input_dim=8)
                )
        model.add(Dropout(hp.Choice('dropout'+str(i),values=[0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9])))
    else:
        model.add(Dense(hp.Int('units'+str(i),min_value=8,max_value=256,step=8),
                        activation=hp.Choice(f'activation_{i}', values=['relu', 'tanh', 'sigmoid']))
                    )
        model.add(Dropout(hp.Choice('dropout'+str(i),values=[0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9])))

    counter+=1
  model.add(Dense(1,activation='sigmoid'))

  model.compile(optimizer=hp.Choice('optimizer',values=['sgd','adam','rmsprop','adadelta']),
                loss='binary_crossentropy',metrics=['accuracy'])
  return model

In [173]:
tuner=kt.RandomSearch(build_model,
                      objective='val_accuracy',
                      max_trials=5,
                      directory='mydir',
                      project_name='Final2')


In [174]:
tuner.search(X_train,y_train,epochs=5,validation_data=(X_test,y_test))

Trial 5 Complete [00h 00m 07s]
val_accuracy: 0.798701286315918

Best val_accuracy So Far: 0.798701286315918
Total elapsed time: 00h 00m 32s


In [175]:
tuner.get_best_hyperparameters()[0].values

{'num_layers': 7,
 'units0': 72,
 'activation_0': 'tanh',
 'dropout0': 0.2,
 'optimizer': 'adam',
 'units1': 168,
 'activation_1': 'tanh',
 'dropout1': 0.4,
 'units2': 184,
 'activation_2': 'relu',
 'dropout2': 0.7,
 'units3': 200,
 'activation_3': 'tanh',
 'dropout3': 0.6,
 'units4': 96,
 'activation_4': 'relu',
 'dropout4': 0.4,
 'units5': 168,
 'activation_5': 'tanh',
 'dropout5': 0.9,
 'units6': 120,
 'activation_6': 'tanh',
 'dropout6': 0.1,
 'units7': 16,
 'activation_7': 'relu',
 'dropout7': 0.9,
 'units8': 40,
 'activation_8': 'sigmoid',
 'dropout8': 0.5,
 'units9': 16,
 'activation_9': 'sigmoid',
 'dropout9': 0.6}

In [176]:
model2=tuner.get_best_models(num_models=1)[0]

/usr/local/lib/python3.13/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 34 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [177]:
model2.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 72)             │           648 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 72)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 168)            │        12,264 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 168)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 184)            │        31,096 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 184)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 200)            │        37,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 200)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 96)             │        19,296 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 96)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 168)            │        16,296 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 168)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 120)            │        20,280 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 120)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 1)              │           121 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 137,001 (535.16 KB)

 Trainable params: 137,001 (535.16 KB)

 Non-trainable params: 0 (0.00 B)

In [180]:
model2.fit(X_train,y_train,epochs=20,batch_size=32,validation_data=(X_test,y_test))

Epoch 1/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 35ms/step - accuracy: 0.7231 - loss: 0.5631 - val_accuracy: 0.7273 - val_loss: 0.7782
Epoch 2/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7264 - loss: 0.5570 - val_accuracy: 0.7143 - val_loss: 0.8209
Epoch 3/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7378 - loss: 0.5423 - val_accuracy: 0.7338 - val_loss: 0.7690
Epoch 4/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7524 - loss: 0.5317 - val_accuracy: 0.7273 - val_loss: 0.8167
Epoch 5/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7166 - loss: 0.5441 - val_accuracy: 0.7143 - val_loss: 0.8703
Epoch 6/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7296 - loss: 0.5410 - val_accuracy: 0.7013 - val_loss: 0.8968
Epoch 7/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.7362 - loss: 0.5368 - val_accuracy: 0.7208 - val_loss: 0.8732
Epoch 8/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7280 - loss: 0.5445 - val_accuracy: 0.7338 - val_los

## Conclusion :
### Mostly Focus on How learn and apply different different DL and Optimizers techniques
Here we get better resluts with multiple techniques like no.of nuerons, no.of layers, diff. - diff.  activations and diff. - diff. loss functions and final with Dropout then we get more generalized accuracy